# Notebook 10 — CICIDS2017 Feature Engineering & Training Preparation

**AEGIS / AI-IDS** · Phase 2: Prepare data for baseline binary classification.

Loads the output of Notebook 09, audits selected features, removes invalid negative rate/duration values (without overwriting the combined source), groups identical five-feature vectors into the same partition to reduce exact-feature leakage, then writes train/validation/test CSVs.

**Important caveats:** Exact-feature grouping does **not** prevent near-duplicate or same-session leakage; CICIDS2017 attack classes are highly correlated with capture days. This notebook prepares a provisional baseline, not a fully independent deployment evaluation. The five live feature formulas and units still need comparison with TShark output. Do not treat test metrics as real-world detection guarantees.


## 1. Imports and project paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ml' / 'data').is_dir()), None)
if root is None:
    raise FileNotFoundError('Cannot find repository root containing ml/data')
input_file = root / 'ml' / 'data' / 'processed' / 'combined' / 'CICIDS2017_Combined_Cleaned.csv'
output_dir = root / 'ml' / 'data' / 'processed' / 'model_ready'
output_dir.mkdir(parents=True, exist_ok=True)
if not input_file.is_file():
    raise FileNotFoundError(f'Run Notebook 09 first: {input_file}')
print('Input:', input_file)
print('Output:', output_dir)


## 2. Load only the columns needed for the first live-compatible baseline

In [ ]:
FEATURES = ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Flow Bytes/s', 'Flow Packets/s']
META = ['SourceDataset', 'SourceRow', 'Label']
requested = FEATURES + META

df = pd.read_csv(input_file, usecols=requested, low_memory=False)
assert set(requested) == set(df.columns), 'Required columns are missing'
df = df[requested]
df['Label'] = df['Label'].astype('string').str.strip()
print('Input shape:', df.shape)
print(df['Label'].value_counts().to_string())
display(df.head())


## 3. Audit numerical anomalies — do not silently change features

In [ ]:
audit = pd.DataFrame({
    'missing': df[FEATURES].isna().sum(),
    'infinite': np.isinf(df[FEATURES].to_numpy(dtype='float64')).sum(axis=0),
    'negative': (df[FEATURES] < 0).sum(),
    'zero': (df[FEATURES] == 0).sum(),
    'min': df[FEATURES].min(),
    'max': df[FEATURES].max(),
})
display(audit)
assert not df[META].isna().any().any(), 'Missing source/target metadata'


## 4. Exclude impossible negative durations and rates from the *model-ready copy*

In [ ]:
finite = np.isfinite(df[FEATURES].to_numpy(dtype='float64')).all(axis=1)
nonnegative = (df[FEATURES] >= 0).all(axis=1)
valid = finite & nonnegative
excluded = df.loc[~valid, 'Label'].value_counts(dropna=False)
print('Excluded rows:', int((~valid).sum()))
print('Excluded by label:')
print(excluded.to_string())
model_df = df.loc[valid].copy().reset_index(drop=True)
assert len(model_df) > 0
assert np.isfinite(model_df[FEATURES].to_numpy(dtype='float64')).all()
assert (model_df[FEATURES] >= 0).all().all()


## 5. Encode binary target; keep original attack class for auditing

In [ ]:
model_df['BinaryLabel'] = (model_df['Label'] != 'BENIGN').astype('int8')
print('0 = BENIGN, 1 = ATTACK')
print(model_df['BinaryLabel'].value_counts().sort_index().to_string())
print('Attack-class counts:')
print(model_df.loc[model_df['BinaryLabel'] == 1, 'Label'].value_counts().to_string())


## 6. Group identical five-feature vectors before the split

In [ ]:
# Identical model-input vectors must not be distributed across train/val/test.
# This is intentionally based only on features, not on label or source.
signature = pd.util.hash_pandas_object(model_df[FEATURES], index=False).to_numpy(dtype='uint64')
model_df['FeatureSignature'] = signature

label_counts_per_group = model_df.groupby('FeatureSignature')['BinaryLabel'].nunique()
conflicting_groups = label_counts_per_group[label_counts_per_group > 1]
print('Rows:', len(model_df))
print('Unique five-feature signatures:', int(model_df['FeatureSignature'].nunique()))
print('Signature groups containing both BENIGN and ATTACK:', len(conflicting_groups))
print('Rows in conflicting groups:', int(model_df['FeatureSignature'].isin(conflicting_groups.index).sum()))
# These are not removed; they should be investigated. Hashes are screening signatures,
# not proof of row identity where collisions are possible.


## 7. Deterministic group-assigned 70/15/15 split

In [ ]:
# Every identical five-feature signature receives the same split.
# A 64-bit hash-to-bucket mapping gives a reproducible approximate split.
# It is NOT a capture-session split; day/session-level evaluation is still needed.
SEED = np.uint64(20261010)
keys = (signature ^ SEED)
# Mix 64-bit signatures to distribute buckets more evenly.
keys = keys ^ (keys >> np.uint64(30))
keys = keys * np.uint64(0xbf58476d1ce4e5b9)
keys = keys ^ (keys >> np.uint64(27))
keys = keys * np.uint64(0x94d049bb133111eb)
keys = keys ^ (keys >> np.uint64(31))
bucket = keys % np.uint64(100)
model_df['Split'] = np.select([bucket < 70, bucket < 85], ['train', 'validation'], default='test')
print(model_df['Split'].value_counts().to_string())
print('\nBinary classes per split:')
print(pd.crosstab(model_df['Split'], model_df['BinaryLabel']).to_string())
print('\nOriginal labels per split:')
display(pd.crosstab(model_df['Label'], model_df['Split']))


## 8. Leakage and class checks

In [ ]:
groups_by_split = {name: set(model_df.loc[model_df['Split'] == name, 'FeatureSignature'])
                   for name in ['train', 'validation', 'test']}
assert not (groups_by_split['train'] & groups_by_split['validation'])
assert not (groups_by_split['train'] & groups_by_split['test'])
assert not (groups_by_split['validation'] & groups_by_split['test'])
for name in ['train', 'validation', 'test']:
    portion = model_df.loc[model_df['Split'] == name]
    assert len(portion) > 0, f'{name} is empty'
    assert portion['BinaryLabel'].nunique() == 2, f'{name} lacks a binary class'
print('Passed: no identical five-feature signatures cross partitions.')
print('Caution: temporal/session correlations and unseen-attack generalization remain unresolved.')


## 9. Export baseline inputs and audit reports

In [ ]:
# No scaling, oversampling or class weights are fitted here.
# Train-derived transforms belong in the model training pipeline.
output_cols = FEATURES + ['BinaryLabel', 'Label', 'SourceDataset', 'SourceRow']
for name in ['train', 'validation', 'test']:
    subset = model_df.loc[model_df['Split'] == name, output_cols]
    target_file = output_dir / f'CICIDS2017_Binary_{name}.csv'
    subset.to_csv(target_file, index=False)
    print(name, len(subset), '->', target_file)

pd.crosstab(model_df['Label'], model_df['Split']).to_csv(output_dir / 'Split_Label_Distribution.csv')
audit.to_csv(output_dir / 'Five_Feature_Anomaly_Audit.csv')
excluded.rename('Excluded_Rows').to_csv(output_dir / 'Excluded_Rows_By_Label.csv')
print('\nNOTE: SourceDataset, SourceRow, Label are metadata only. Do not train on them.')
print('Notebook 10 complete. Next: train a binary baseline using FEATURES + BinaryLabel only.')
